In [13]:
import os
from pathlib import Path
from typing import final

from dotenv import load_dotenv
from openai import OpenAI
from src.kg.neo4j_client import get_neo4j_database, get_neo4j_driver
from src.rag.retrieve import retrieve_candidates_graph
from src.rag.merge import merge_candidates
from src.rag.planner import plan_query
from src.rag.composer import compose_answer

load_dotenv()

In [14]:
# --- Configuration ---
PROJECT_ROOT = Path(".").resolve().parent

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Missing OPENAI_API_KEY env variable")

openai_client = OpenAI(api_key=api_key)
driver = get_neo4j_driver()
database = get_neo4j_database()

# Configs
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-large")
INDEX_NAME = os.getenv("NEO4J_VECTOR_INDEX", "offer_embedding_index")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o")

In [15]:
def run_search(user_query: str):
    print(f"🔎 Analyzing: '{user_query}'...")

    # PLAN
    plan = plan_query(openai_client, user_query, PROJECT_ROOT)
    print(f"📋 Plan: {plan}")


    # RETRIEVE
    all_hits = []
    for rq in plan.rewritten_queries:
        hits = retrieve_candidates_graph(
            driver=driver,
            database=database,
            openai_client=openai_client,
            embedding_model=EMBEDDING_MODEL,
            index_name=INDEX_NAME,
            query=rq,
            top_k=12,
            cities=plan.cities,
            categories=plan.categories,
            segment_types=plan.segment_types,
            product_codes=plan.product_codes,
        )
        all_hits.extend(hits)

    # MERGE & RANK
    merged = merge_candidates(all_hits)
    final_offers = merged[:7]

    if not final_offers:
        return "❌ No relevant offers found for your criteria."

    # GENERATE
    print(f"🤖 Generating answer based on users promt")
    answer = compose_answer(
        client=openai_client,
        user_query=user_query,
        offers=final_offers,
        model=CHAT_MODEL,
    )

    return answer



In [17]:
query = "ბათუმში მივდივარ ივენთზე, მჭირდება შესაბამისი ტანსაცმელი და სასტუმრო"

response = run_search(query)

print("-" * 50)
print(response)

🔎 Analyzing: 'ბათუმში მივდივარ ივენთზე, მჭირდება შესაბამისი ტანსაცმელი და სასტუმრო'...
📋 Plan: cities=['ბათუმი'] categories=['შოპინგი', 'დასვენება'] segment_types=[] product_codes=[] rewritten_queries=['ივენთებისთვის ტანსაცმელი ბათუმში', 'სასტუმრო ბათუმში', 'შოპინგი ბათუმში']
🤖 Generating answer based on users promt
--------------------------------------------------
გამარჯობა, მე ვარ საქართველოს ბანკის შეთავაზებების ასისტენტი და მზად ვარ დაგეხმაროთ საუკეთესო შეთავაზებების პოვნაში ბათუმში თქვენი ივენთისთვის.

თუ გსურთ კომფორტული და ელეგანტური სასტუმრო, გირჩევთ შერატონ ბათუმს, სადაც SOLO ბარათის მფლობელებისთვის 20%-იანი ფასდაკლება მოქმედებს. ეს არის შესანიშნავი შანსი, რომ ისიამოვნოთ მაღალი დონის მომსახურებით და დასვენებით. დამატებითი ინფორმაცია შეგიძლიათ იხილოთ აქ: https://bankofgeorgia.ge/ka/offers-hub/details/11732

თუ უფრო თანამედროვე სტილის სასტუმრო გსურთ, იბის სტაილს ბათუმი გთავაზობთ 25%-იან ფასდაკლებას SOLO ბარათის მფლობელებისთვის. ეს არის შესანიშნავი არჩევანი მათთვის, ვინც ეძებს კ